In [2]:
# charts.py
import matplotlib.pyplot as plt
import seaborn as sns
import os
import zipfile

# Global professional aesthetic styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'font.size': 9, 'axes.labelsize': 10, 'axes.titlesize': 11})

def save_zip(*figs):
    """Saare plots ko locally save karke single zip deliverable banane ka utility function"""
    os.makedirs("charts", exist_ok=True)
    paths = []

    for i, fig in enumerate(figs, start=1):
        path = f"charts/chart_{i}.png"
        fig.savefig(path, bbox_inches="tight")
        paths.append(path)

    zip_path = "charts/all_charts.zip"
    with zipfile.ZipFile(zip_path, "w") as z:
        for p in paths:
            z.write(p)
    return zip_path

def generate_all_plots(temp, area_col):
    plots = {}
    
    # Core reference variables optimization fallback
    lat_col = next((c for c in temp.columns if "LAT" in c.upper()), None)
    lon_col = next((c for c in temp.columns if "LON" in c.upper()), None)

    # --- 1. PIE CHART ---
    fig1, ax1 = plt.subplots(figsize=(5, 4))
    if "Victim Sex" in temp.columns:
        pie_data = temp["Victim Sex"].replace("Unknown", plt.cm.colors.to_rgba('white', alpha=0)).dropna()
        if len(pie_data) > 0:
            pie_data.value_counts().plot.pie(autopct="%1.1f%%", ax=ax1)
    ax1.set_title("Gender Distribution (Pie)")
    plots['plot_1'] = fig1

    # --- 2. COUNT PLOT ---
    fig2, ax2 = plt.subplots(figsize=(5, 4))
    if "Victim Sex" in temp.columns:
        count_data = temp["Victim Sex"].replace("Unknown", plt.cm.colors.to_rgba('white', alpha=0)).dropna()
        if len(count_data) > 0:
            sns.countplot(x=count_data, order=count_data.value_counts().index, ax=ax2)
    ax2.set_title("Victim Count by Gender")
    plots['plot_2'] = fig2

    # --- 3. VIOLIN PLOT ---
    fig3, ax3 = plt.subplots(figsize=(5, 4))
    if "Victim Sex" in temp.columns and "Victim Age" in temp.columns:
        clean = temp[["Victim Sex", "Victim Age"]].dropna()
        clean = clean[clean["Victim Sex"] != "Unknown"]
        if len(clean) > 0:
            sns.violinplot(x="Victim Sex", y="Victim Age", data=clean, ax=ax3)
    ax3.set_title("Age Density Distribution by Sex")
    plots['plot_3'] = fig3

    # --- 4. HISTOGRAM ---
    fig4, ax4 = plt.subplots(figsize=(5, 4))
    if "Victim Age" in temp.columns:
        ax4.hist(temp["Victim Age"].dropna(), bins=25, color="skyblue", edgecolor="black")
    ax4.set_title("Victim Age Frequency Distribution")
    plots['plot_4'] = fig4

    # --- 5. BAR CHART ---
    fig5, ax5 = plt.subplots(figsize=(5, 4))
    if area_col:
        temp[area_col].value_counts().head(10).plot(kind="bar", ax=ax5, color="coral")
    ax5.set_title("Top 10 High Accident Areas")
    plt.xticks(rotation=45, ha="right")
    plots['plot_5'] = fig5

    # --- 6. LINE CHART ---
    fig6, ax6 = plt.subplots(figsize=(5, 4))
    if "Year" in temp.columns:
        temp.groupby("Year").size().plot(ax=ax6, marker='o', color='green')
    ax6.set_title("Year-over-Year Collision Trend")
    plots['plot_6'] = fig6

    # --- 7. SCATTER PLOT ---
    fig7, ax7 = plt.subplots(figsize=(5, 4))
    if "Victim Age" in temp.columns and lat_col:
        clean = temp[[lat_col, "Victim Age"]].dropna()
        if len(clean) > 0:
            ax7.scatter(clean["Victim Age"], clean[lat_col], alpha=0.5, color='purple')
    ax7.set_title("Scatter: Victim Age vs Latitude Geographic")
    plots['plot_7'] = fig7

    # --- 8. HEATMAP ---
    fig8, ax8 = plt.subplots(figsize=(5, 4))
    numeric = temp.select_dtypes(include=["int64", "float64"])
    if not numeric.empty and numeric.shape[1] > 1:
        sns.heatmap(numeric.corr(), cmap="coolwarm", annot=False, ax=ax8)
    ax8.set_title("Numerical Attributes Correlation Heatmap")
    plots['plot_8'] = fig8

    # --- 9. BOX PLOT (Mandatory Chart 9) ---
    fig9, ax9 = plt.subplots(figsize=(5, 4))
    if "Victim Age" in temp.columns:
        sns.boxplot(x=temp["Victim Age"], ax=ax9, color="lightpink")
    ax9.set_title("Victim Age Range Spread & Outliers Detection")
    plots['plot_9'] = fig9

    # --- 10. AREA CHART (Mandatory Chart 10) ---
    fig10, ax10 = plt.subplots(figsize=(5, 4))
    if "Month" in temp.columns:
        monthly_counts = temp.groupby("Month").size()
        if not monthly_counts.empty:
            ax10.fill_between(monthly_counts.index, monthly_counts.values, color="teal", alpha=0.4)
            ax10.plot(monthly_counts.index, monthly_counts.values, color="teal", marker='s')
            ax10.set_xticks(range(1, 13))
    ax10.set_title("Cumulative Incident Trends Across Months")
    plots['plot_10'] = fig10

    # Execute dynamic layout spacing adjustment logic cleanly
    for f in plots.values():
        f.tight_layout()

    return plots